# Notebook 02 — Feature Engineering

Loads the position CSVs produced by Notebook 01, computes derived features (velocity magnitude, orbital period estimate, time-of-day encoding), and cuts the time series into sliding-window sequence pairs ready for model training.

**Input:** `data/collected/positions_<NORAD>.csv` (or upload ZIP from Notebook 01 on Colab)

**Output:** `data/collected/dataset_X.npy` and `data/collected/dataset_y.npy`

- `X` shape: `(N, 90, F)` — N sequences, 90-minute input window, F features per step
- `y` shape: `(N, 4, 3)` — targets at T+10, T+30, T+60, T+120 min, each as (lat, lon, alt)

**Works on:** local machine, Google Colab (no GPU needed)

---

### What this notebook does
1. Load position CSVs (or upload ZIP on Colab)
2. Compute derived features per timestep
3. Build sliding windows with configurable input length and forecast horizons
4. Normalise features with a per-feature StandardScaler
5. Split into train / val / test by time (no random shuffle)
6. Save arrays and the scaler to `data/collected/`

## 0 — Environment Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running on Google Colab')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'scikit-learn', 'plotly'], check=True)
else:
    print('Running locally')

import os, math, zipfile, io
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.preprocessing import StandardScaler
import pickle

pio.renderers.default = 'colab' if IN_COLAB else 'notebook_connected'
DATA_DIR = Path('/content/collected') if IN_COLAB else Path('../data/collected')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data directory: {DATA_DIR.resolve()}')
print('All imports OK')

Running on Google Colab
Data directory: /content/collected
All imports OK


## 1 — Load Data

**Colab:** upload the `collected_positions.zip` downloaded from Notebook 01.  
**Local:** CSVs are loaded automatically from `data/collected/`.

In [2]:
if IN_COLAB:
    _already_present = list(DATA_DIR.glob('positions_*.csv'))
    if not _already_present:
        from google.colab import files as _colab_files
        print('Upload the collected_positions.zip file from Notebook 01.')
        _uploaded = _colab_files.upload()
        if _uploaded:
            _zip_bytes = next(iter(_uploaded.values()))
            with zipfile.ZipFile(io.BytesIO(_zip_bytes)) as zf:
                zf.extractall(DATA_DIR)
            print(f'  Extracted to {DATA_DIR}')
    else:
        print(f'  {len(_already_present)} CSV(s) already in {DATA_DIR}')

csv_files = sorted(DATA_DIR.glob('positions_*.csv'))
print(f'Found {len(csv_files)} CSV file(s): {[f.name for f in csv_files]}')

STALE_COLS = ['sgp4_stale_lat', 'sgp4_stale_lon', 'sgp4_stale_alt']

dfs = []
for f in csv_files:
    df = pd.read_csv(f, parse_dates=['timestamp_utc'])
    norad_id = int(f.stem.split('_')[1])
    df['norad_id'] = norad_id
    # Fill any rows with no stale baseline (first STALE_DAYS of data) with NaN
    for c in STALE_COLS + ['sgp4_stale_age']:
        if c not in df.columns:
            df[c] = np.nan
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True).sort_values('timestamp_utc').reset_index(drop=True)
n_stale = combined['sgp4_stale_lat'].notna().sum()
print(f'Total records: {len(combined):,}  |  With stale baseline: {n_stale:,}  |  Satellites: {combined["norad_id"].nunique()}')

Upload the collected_positions.zip file from Notebook 01.


Saving collected_positions.zip to collected_positions.zip
  Extracted to /content/collected
Found 4 CSV file(s): ['positions_20580.csv', 'positions_25544.csv', 'positions_33591.csv', 'positions_43017.csv']
Total records: 1,052,616  |  With stale baseline: 1,035,307  |  Satellites: 4


## 2 — Feature Engineering

Features per timestep:

| Feature | Description |
|---|---|
| `lat_deg`, `lon_deg`, `alt_km` | Raw geodetic position |
| `vx, vy, vz` | Velocity components (km/s) |
| `speed_km_s` | Total velocity magnitude |
| `orbital_period_min` | Estimated from altitude: $T = 2\pi\sqrt{(R_E + h)^3 / \mu}$ |
| `sin_lat`, `cos_lat` | Cyclic encoding of latitude |
| `sin_lon`, `cos_lon` | Cyclic encoding of longitude |
| `hour_sin`, `hour_cos` | Time-of-day cyclic encoding |
| `tle_age_days` | Age of the TLE used at this timestep |

In [3]:
MU_KM3 = 398600.4418  # Earth gravitational parameter km^3/s^2
RE_KM  = 6371.0       # mean Earth radius km

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Velocity magnitude
    df['speed_km_s'] = np.sqrt(df['vx_km_s']**2 + df['vy_km_s']**2 + df['vz_km_s']**2)
    # Estimated orbital period (minutes)
    r_km = RE_KM + df['alt_km']
    df['orbital_period_min'] = 2 * math.pi * np.sqrt(r_km**3 / MU_KM3) / 60.0
    # Cyclic encoding of lat/lon to avoid discontinuity at +/-180 and poles
    df['sin_lat'] = np.sin(np.radians(df['lat_deg']))
    df['cos_lat'] = np.cos(np.radians(df['lat_deg']))
    df['sin_lon'] = np.sin(np.radians(df['lon_deg']))
    df['cos_lon'] = np.cos(np.radians(df['lon_deg']))
    # Time-of-day cyclic encoding
    hour_frac = df['timestamp_utc'].dt.hour + df['timestamp_utc'].dt.minute / 60.0
    df['hour_sin'] = np.sin(2 * math.pi * hour_frac / 24.0)
    df['hour_cos'] = np.cos(2 * math.pi * hour_frac / 24.0)
    return df

combined = add_features(combined)

FEATURE_COLS = [
    'lat_deg', 'lon_deg', 'alt_km',
    'vx_km_s', 'vy_km_s', 'vz_km_s', 'speed_km_s',
    'orbital_period_min',
    'sin_lat', 'cos_lat', 'sin_lon', 'cos_lon',
    'hour_sin', 'hour_cos',
    'tle_age_days',
]
TARGET_COLS = ['lat_deg', 'lon_deg', 'alt_km']

print(f'Feature count: {len(FEATURE_COLS)}')
print(combined[FEATURE_COLS].describe().round(3))

Feature count: 15
           lat_deg      lon_deg       alt_km      vx_km_s      vy_km_s  \
count  1052616.000  1052616.000  1052616.000  1052616.000  1052616.000   
mean        -0.065       -0.091      577.798       -0.000       -0.000   
std         40.998      103.914      172.739        4.323        4.327   
min        -82.566     -180.000      413.293       -7.715       -7.668   
25%        -28.515      -90.233      438.915       -3.694       -3.709   
50%         -0.028       -0.062      480.455       -0.001        0.001   
75%         28.445       89.863      839.512        3.704        3.704   
max         82.567      180.000      886.450        7.670        7.693   

           vz_km_s   speed_km_s  orbital_period_min      sin_lat      cos_lat  \
count  1052616.000  1052616.000         1052616.000  1052616.000  1052616.000   
mean         0.007        7.575              96.100       -0.001        0.767   
std          4.470        0.100               3.601        0.593        

## 3 — Sliding Window Construction

Input window: 90 minutes (90 timesteps at 1-min resolution)  
Forecast horizons: T+10, T+30, T+60, T+120 minutes

In [4]:
INPUT_LEN     = 90    # input steps (minutes)
HORIZONS      = [10, 30, 60, 120]  # forecast horizons (minutes ahead)
MAX_GAP_MIN   = 3    # max allowable gap between consecutive records
STEP          = 5    # slide every 5 steps to reduce redundancy

def build_windows(df_sat: pd.DataFrame, feat_cols, tgt_cols, stale_cols,
                  input_len, horizons, max_gap_min, step):
    """
    Build sliding windows. Returns:
      X            (N, input_len, F)  — normalised features
      y            (N, H, 3)          — ground-truth lat/lon/alt at each horizon
      y_sgp4_stale (N, H, 3)          — SGP4-stale prediction at each horizon (NaN if unavailable)
    """
    vals   = df_sat[feat_cols].values.astype(np.float32)
    tgts   = df_sat[tgt_cols].values.astype(np.float32)
    stale  = df_sat[stale_cols].values.astype(np.float32)  # NaN where no stale available
    times  = df_sat['timestamp_utc'].values
    max_h  = max(horizons)

    X_list, y_list, y_stale_list = [], [], []

    for start in range(0, len(vals) - input_len - max_h, step):
        window_times = times[start: start + input_len]
        diffs = np.diff(window_times.astype('datetime64[m]')).astype(float)
        if diffs.max() > max_gap_min:
            continue

        y_steps, y_stale_steps = [], []
        ok = True
        for h in horizons:
            idx = start + input_len + h - 1
            if idx >= len(tgts):
                ok = False; break
            y_steps.append(tgts[idx])
            y_stale_steps.append(stale[idx])
        if not ok:
            continue

        X_list.append(vals[start: start + input_len])
        y_list.append(np.stack(y_steps, axis=0))         # (H, 3)
        y_stale_list.append(np.stack(y_stale_steps, axis=0))  # (H, 3)

    return (np.array(X_list),
            np.array(y_list),
            np.array(y_stale_list))

all_X, all_y, all_y_stale = [], [], []
for norad_id, grp in combined.groupby('norad_id'):
    grp = grp.sort_values('timestamp_utc').reset_index(drop=True)
    X, y, y_stale = build_windows(grp, FEATURE_COLS, TARGET_COLS, STALE_COLS,
                                   INPUT_LEN, HORIZONS, MAX_GAP_MIN, STEP)
    n_valid_stale = (~np.isnan(y_stale[:, 0, 0])).sum()
    print(f'  NORAD {norad_id}: {len(X):,} windows  ({n_valid_stale:,} with stale baseline)')
    all_X.append(X); all_y.append(y); all_y_stale.append(y_stale)

X_all       = np.concatenate(all_X,       axis=0)
y_all       = np.concatenate(all_y,       axis=0)
y_stale_all = np.concatenate(all_y_stale, axis=0)
print(f'\nX shape         : {X_all.shape}')
print(f'y shape         : {y_all.shape}')
print(f'y_sgp4_stale    : {y_stale_all.shape}  ({(~np.isnan(y_stale_all[:,0,0])).sum():,} non-NaN)')

  NORAD 20580: 52,554 windows  (51,708 with stale baseline)
  NORAD 25544: 52,659 windows  (51,813 with stale baseline)
  NORAD 33591: 52,599 windows  (51,754 with stale baseline)
  NORAD 43017: 52,544 windows  (51,698 with stale baseline)

X shape         : (210356, 90, 15)
y shape         : (210356, 4, 3)
y_sgp4_stale    : (210356, 4, 3)  (206,973 non-NaN)


## 4 — Normalise and Split

Train / Val / Test split is **time-ordered** (no shuffle) to prevent data leakage.  
Split: 70% train, 15% validation, 15% test.

In [5]:
N = len(X_all)
n_train = int(N * 0.70)
n_val   = int(N * 0.15)

X_train,       y_train,       y_stale_train = X_all[:n_train],             y_all[:n_train],             y_stale_all[:n_train]
X_val,         y_val,         y_stale_val   = X_all[n_train:n_train+n_val], y_all[n_train:n_train+n_val], y_stale_all[n_train:n_train+n_val]
X_test,        y_test,        y_stale_test  = X_all[n_train+n_val:],        y_all[n_train+n_val:],        y_stale_all[n_train+n_val:]

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')

# Fit scaler ONLY on training data
scaler_X = StandardScaler()
scaler_X.fit(X_train.reshape(-1, X_train.shape[-1]))

def scale_X(arr):
    shape = arr.shape
    return scaler_X.transform(arr.reshape(-1, shape[-1])).reshape(shape)

X_train_s = scale_X(X_train)
X_val_s   = scale_X(X_val)
X_test_s  = scale_X(X_test)
print('Normalisation done.')

# Save arrays
np.save(DATA_DIR / 'X_train.npy',          X_train_s)
np.save(DATA_DIR / 'y_train.npy',          y_train)
np.save(DATA_DIR / 'X_val.npy',            X_val_s)
np.save(DATA_DIR / 'y_val.npy',            y_val)
np.save(DATA_DIR / 'X_test.npy',           X_test_s)
np.save(DATA_DIR / 'y_test.npy',           y_test)
np.save(DATA_DIR / 'y_sgp4_stale_train.npy', y_stale_train)
np.save(DATA_DIR / 'y_sgp4_stale_val.npy',   y_stale_val)
np.save(DATA_DIR / 'y_sgp4_stale_test.npy',  y_stale_test)

with open(DATA_DIR / 'scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)

import json
meta = {'feature_cols': FEATURE_COLS, 'target_cols': TARGET_COLS,
        'stale_cols': STALE_COLS, 'horizons': HORIZONS, 'input_len': INPUT_LEN}
with open(DATA_DIR / 'dataset_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('All arrays and metadata saved.')

Train: 147,249  Val: 31,553  Test: 31,554
Normalisation done.
All arrays and metadata saved.


## 5 — Download Dataset (Colab only)

Download the `.npy` files and `scaler_X.pkl` to upload in Notebook 03.

In [6]:
if IN_COLAB:
    from google.colab import files as _colab_files
    zip_buf = io.BytesIO()
    with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fname in ['X_train.npy', 'y_train.npy', 'X_val.npy', 'y_val.npy',
                      'X_test.npy',  'y_test.npy',
                      'y_sgp4_stale_train.npy', 'y_sgp4_stale_val.npy', 'y_sgp4_stale_test.npy',
                      'scaler_X.pkl', 'dataset_meta.json']:
            p = DATA_DIR / fname
            if p.exists():
                zf.write(p, fname)
    zip_buf.seek(0)
    with open('/content/dataset.zip', 'wb') as f:
        f.write(zip_buf.read())
    _colab_files.download('/content/dataset.zip')
    print('Download started — save this ZIP, you will upload it in Notebook 03.')
else:
    print(f'Local: arrays saved to {DATA_DIR.resolve()}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started — save this ZIP, you will upload it in Notebook 03.
